## Word Embedding

简单规划一下数据集放在哪里

In [3]:
from pathlib import Path
from collections import Counter
import zipfile

import numpy as np
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
TEXT8_ZIP = DATA_DIR / "text8.zip"
TEXT8_TXT = DATA_DIR / "text8"

CORPUS_WORDS = 3_000_000
MIN_COUNT = 5
SUBSAMPLE_T = 1e-5

rng = np.random.default_rng(42)

下载语料

In [4]:
if not TEXT8_TXT.exists():
    if not TEXT8_ZIP.exists():
        print("Downloading...")
        with requests.get("http://mattmahoney.net/dc/text8.zip",
                          stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(TEXT8_ZIP,"wb") as f:
                for chunk in r.iter_content(1<<20):
                    f.write(chunk)
    with zipfile.ZipFile(TEXT8_ZIP) as zf:
        zf.extractall(DATA_DIR)

text = TEXT8_TXT.read_text(encoding="utf-8")
print("total:",f"{len(text):,}")
print(text[:300])

Downloading...
total: 100,000,000
 anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term is still used in a pejorative way to describe any act that used violent means to destroy the organiz


分词

In [5]:
words = text.split()[:CORPUS_WORDS]

print("token 总数:", f"{len(words):,}")
print("前30个词:", words[:30])
print("不重复的词:", f"{len(set(words)):,}")

print(type(words))
print(type(words[0]))

token 总数: 3,000,000
前30个词: ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english', 'revolution', 'and', 'the', 'sans', 'culottes', 'of', 'the', 'french', 'revolution', 'whilst']
不重复的词: 96,644
<class 'list'>
<class 'str'>


数词频

In [9]:
counter = Counter(words)
print(f"{'词':<10}{'次数':>10}{'占比':>10}")
for w,c in counter.most_common(15):
    print(f"{w:<10}{c:>10}{c/len(words):>10.2%}")
print("\n只出现 1 次的词:", sum(1 for c in counter.values() if c == 1))
print("出现 <5 次的单词:", sum(1 for c in counter.values() if c < 5))

rare_tokens = sum(c for c in counter.values() if c < 5)
print(f"{rare_tokens:,} 个 token 占 {rare_tokens/len(words):.2%}")

词                 次数        占比
the           190150     6.34%
of            107283     3.58%
and            75482     2.52%
one            74263     2.48%
in             66680     2.22%
a              56730     1.89%
to             55219     1.84%
zero           47433     1.58%
nine           42802     1.43%
two            35183     1.17%
is             32111     1.07%
as             23027     0.77%
eight          22952     0.77%
three          21363     0.71%
was            21080     0.70%

只出现 1 次的词: 44486
出现 <5 次的单词: 69323
109,992 个 token 占 3.67%


建词表，砍掉低频词

In [11]:
kept = [(w, c) for w, c in counter.most_common() if c >= MIN_COUNT]
idx2word = [w for w, _ in kept]
word2idx = {w: i for i, w in enumerate(idx2word)}

print(f"原始词表 {len(counter):,} -> 过滤后 {len(idx2word):,}")
print("被丢掉的词举例:", [w for w, c in counter.items() if c < MIN_COUNT][:20])
print("前 10 个 id 对应的词", idx2word[:10]) 

原始词表 96,644 -> 过滤后 27,321
被丢掉的词举例: ['diggers', 'culottes', 'nihilism', 'anomie', 'harmonious', 'stoic', 'omnipotence', 'regimentation', 'levellers', 'communistic', 'lahontan', 'nouveaux', 'rique', 'septentrionale', 'anarchiste', 'hurled', 'girondins', 'propri', 'mutualism', 'mutuellisme']
前 10 个 id 对应的词 ['the', 'of', 'and', 'one', 'in', 'a', 'to', 'zero', 'nine', 'two']


词变成 id

In [12]:
ids = np.array([word2idx[w] for w in words if w in word2idx],dtype=np.int32)

print(f"编码前{len(words):,} -> 编码后{len(ids):,}")
print("前 20 个 id:", ids[:20])
print("还原回词:", [idx2word[i] for i in ids[:20]])

编码前3,000,000 -> 编码后2,890,008
前 20 个 id: [2067 2763   11    5  190    1 4159   47   58  143  123  807  531 7358
  140    0    1    0   96  961]
还原回词: ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'of', 'the', 'english', 'revolution']
